## Anatomy of a Car

You design a new car by specifying a series of sizes. They are grouped into the following categories:

1. the frame length (total length of the car);
2. the shape of the upper profile;
3. the shape of the lower profile;
4. the first wheel;
5. the second wheel.

See the picture below for reference:

<img src="http://sales.bio.unipd.it/assets/carsim/car_model.png" width="600px">

### 1. Frame Length

This is a single floating point number representing the entire length of the car, from the front to the rear. The allowed range is comprised between 2.0 and 6.0 meters.

### 2. Upper Shape

The upper profile of the car is described by 5 floating point numbers. Each of them describes the height of the frame at equi‑spaced points along the car, from left (back) to right (front).

Example: the array [0.5, 0.7, 1.3, 0.8, 0.5] would correspond to the picture above.

### 3. Lower Shape

The lower profile of the car is described by 5 floating point numbers. Each of them describes the height of the frame at equi‑spaced points along the car, from left (back) to right (front).

Example: the array [0.5, 0.7, 0.4, 0.6, 0.5] would correspond to the picture above.

### Shape Constraints

For manufacturing purposes, the car cannot be either too high or too thin. In numbers, this mean that the **total** height (lower shape + upper shape at any of the 5 provided points) must be comprised between 0.5 and 5.0.  

Example: the sum of the lower and the upper shapes shown before is

`[0.5, 0.7, 1.3, 0.8, 0.5] + [0.5, 0.7, 0.4, 0.6, 0.5] = [1.0, 1.4, 1.7, 1.4, 1.0]`

This car respects the manufacturing constraints.

### 4. Left Wheel

The left wheel is defined by two numbers.

1. its position along the car frame. A real number between 0.0 (wheel at the extreme left, back of the car) and 1.0 (wheel at the extreme right, front of the car);
2. its radius (between 0 and 2 meters).

Example: the left wheel of the picture above is defined as [0.1, 0.5].

### 5. Right Wheel

The right wheel is defined by two numbers.

1. its position along the car frame. A real number between 0.0 (wheel at the extreme left, back of the car) and 1.0 (wheel at the extreme right, front of the car);
2. its radius (between 0 and 2 meters).

Example: the right wheel of the picture above is defined as [0.9, 0.8].

# Simulator

To make the car run along a track, we will use a simulator provided by the test driving team.

You will need to import a few libraries.

In [1]:
import carsim
import numpy as np

Simulator started.

Now open the webpage at this address in a separate browser window: http://localhost:60505/


Do as the text above tells you. Open that URL in a **separate** browser window (right click on the link and select "Open in a new window").

From this point on you have access to three useful functions.

### carsim.simulate

This function takes the following arguments: `frame_length, upper_shape, lower_shape, left_wheel, right_wheel`. That is, the spec of a car.

It then build the vehicle and brings it to the test track. You can follow the test drive in the other browser window (titled "Car Simulator Dashboard").

### carsim.race

Takes a list of car blueprints. Each car is represented as a tuple of the form:

```(frame_length, upper_shape, lower_shape, left_wheel, right_wheel)```

The function will accept any number of cars between 1 and 100 (included).

It then starts a race among all the cars, measuring how well each performed on the track.

The return value `res` is a N by 2 matrix (where N is the number of cars).
* the first column contains the fraction of the track that was covered by the car (0.0 meaning the car didn't even start; 1.0 meaning that the car reached the finish line);
* the second column measures how many seconds it took to the car to finish its race.

Example: res[2, 1] is how long the race of the third car lasted.

<div class="alert alert-warning" style="margin-top:1em">
The race <strong>won't</strong> be shown in the Simulator Dashboard.
</div>

### carsim.new_track

If you don't like the race track, you can change the location with this function. The new track will be used for the following simulations and races.

## Your Turn

It is now time to design the best possible car for offroad racing.

Develop a genetic algorithm for testing out different random designs and to pick the best performing car.

You may find the following functions useful:

* *np.random.rand*: generate a random integer in the interval [0, 1).

In [2]:
np.random.rand()


0.6661831299342066

If you specify a length, it will return that many random numbers.

In [3]:
np.random.rand(10)

array([0.19157872, 0.37089174, 0.24160611, 0.3097555 , 0.7862116 ,
       0.49206872, 0.14706498, 0.53322955, 0.07199572, 0.82476257])

* np.random.rand(): generate a random number *normally distributed* with mean 0 and variance 1.

In [4]:
np.random.randn(5)

array([-1.95709554, -0.59081116, -0.14609382,  1.08454933, -0.06900547])

* *np.min* and *np.max*: compute the minimum / maximum of a sequence of numbers.

In [5]:
np.min(np.arange(10))

0

In [6]:
np.max([1, 3, 2, 0])

3

* *np.clip*: clip values to a given range.

    Given a value `v`, a lower bound `lower` and an upper bound `upper`, makes sure that `lower <= v <= upper`. 

In [7]:
np.clip(3,  # value
        0,  # lower bound
        4)  # upper bound

3

In [8]:
np.clip(-1, 0, 4)

0

In [9]:
np.clip(5, 0, 4)

4

In [10]:
values = np.arange(10)
values

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [11]:
np.clip(values, 3, 5)

array([3, 3, 3, 3, 4, 5, 5, 5, 5, 5])

By default, `np.clip` returns a new array. It is possible to overwrite the original (input) array with the following syntax.

In [12]:
np.clip(values, 3, 5, out=values)
values

array([3, 3, 3, 3, 4, 5, 5, 5, 5, 5])

* *np.argsort*: return the positions of the elements of an array from the smallest to the largest. 

In [13]:
numbers = np.array([3., 1., 2])
np.argsort(numbers)

array([1, 2, 0])

This means that `numbers[1]` has the smallest value; then comes `numbers[2]`; and finally `numbers[0]`, which has also the largest value.

## Implementation

### Step 1

To develop our genetic algorithm for cars, we first need to decide how to represent their genome. Remember that the  blueprint of a car is defined by:
* length;
* upper shape;
* lower shape;
* left wheel position / size;
* right wheel position / size.

For instance:

In [14]:
blueprint = [
    4.,  # length
    np.array([.8, 1., 1., .5, .3]),  # upper profile
    np.array([.3, .3, .3, .3, .3]),  # lower profile
    np.array([.1, .5]),  # left wheel (position, size)
    np.array([.9, .5]),  # right wheel (position, size)
]

This is not a genome, though. A genome is simply a flat sequence of numbers, just like an array. Example:

In [15]:
genome = np.array([4., 1., 1., 1., 1., 1., .5, .5, .5, .5, .5, .1, 1., .9, 1.])

Develop a pair of functions to:
1. convert a blueprint like the above into the corresponding genome;
1. convert the genome back into a blueprint.

In [16]:
def car_genome(car):
    g=[]
    for i in range(0,len(car)):
        if 'float' in str(type(car[i])):
            g.append(car[i])
            
        elif 'array' in str(type(car[i])):
            for j in range(len(car[i])):
                g.append(car[i][j])
                
    genome=np.array(g)
    return genome

In [17]:
def car_from_genome(genome):
    car=[]
    if len(genome) == 15:
        car.append(genome[0]) ##length
        car.append( np.array( [genome[i] for i in range(1,6)] ))  #upper profile
        car.append( np.array( [genome[i] for i in range(6,11)] ))  #lower profile
        car.append( np.array( [genome[i] for i in range(11,13)] ))  #left wheel
        car.append( np.array( [genome[i] for i in range(13,15)] ))  #right wheel
    else:
        print('genome must be of size 15')
    return car

### Step 2

Initially, a genetic algorithm has no idea of which design would work the best for a given task. It creates blueprint totally at random.

Complete the function below, which should return a random car blueprint.

**Note**: not all random combinations of numbers are acceptable. For instance, the frame length must be comprised between 2 and 6 meters.

In [18]:
import random

2<car[0]<6 #frame width
0.5<car[1]+car[2]<5 #total height (sum the arrays elementwise, obtain array of same lenght
0<wheels radius<2

In [19]:
def random_car():
    r=np.zeros(15)
    r[0]=random.uniform(2,6) #frame
    r[1:6]=random.uniform(0.5,5) #upper
    for e in range(1,6):  #lower
        r[e+5]=r[e]-random.uniform(0.5,r[e])
        if r[e+5]+r[e]>5:
            r[e+5]=5-r[e]
    r[11]=random.uniform(0.0,0.6)
    r[12]=random.uniform(0.1,2)
    r[13]=random.uniform(0.6,1.0)
    r[14]=random.uniform(0.1,2)
    
    c=car_from_genome(r)
    return c

Now generate a random car and simulate it.

**Note**: what should you write in place of the ellipses below? Can you use the variable `car1` directly? Why? Is there any way to make it work?

In [20]:
car1 = random_car()
carsim.simulate(car1[0],car1[1],car1[2],car1[3],car1[4])

### Step 3

We need to evaluate the fitness of our car, that is how good it was at completing the track.

Try the following:

In [21]:
carsim.race([car1])

array([[0.0850029, 6.83333  ]])

Here we are making a (boring) race with a single car. The interesting thing is that the `race` function returned two useful numbers:
1. the fraction of the track that was completed by the car (0 is bad, 1 is good);
1. the race time (as long as the car completed the track, lower means faster and thus better).

If you make two cars race, you will get a pair of numbers for each car.

In [22]:
car2 = random_car()
carsim.race([car1, car2])

array([[0.0850029, 6.83333  ],
       [0.0482436, 4.63333  ]])

### Step 4

Things are starting to take shape. We now have a way to generate random cars and function to evaluate their respective fitness.

The next logical step would be to generate an initial population of random cars (say 50) and then to identify the most primising ones.

Complete the following functions:

In [23]:
def initial_population(size):
    cars=list()
    for i in range(size):
        c=random_car()
        cars.append(c)
    return cars

### Step 5

If you make your car population race, you will find somewhat difficult to find the best cars. Your task would probably be easier if the score returned by `carsim.race` would actually consist in a single number. You could simply sort everything from the largest value to the smallest. Here, on the other hand, you got two values for each car (fraction of the track covered, elapsed time).

Leaving programming on the side for a moment, can you think of any reasonable approach to combine those two numbers? Would it make sense to simply sum the two of them? Is any of the two numbers redundant? Alternatively, is that the case only under certain conditions?

When you have found a solution to this problem, complete the function below. `combined_score` should take the output of `carsim.race` and return an array of single‑number scores, one for each car.

In [24]:
def combined_scores(scores):
    combined=list()
    for i in range(len(scores)):
        if scores[i][0] <0.95:
            combined.append(scores[i][0]) #path traveled
        elif scores[i][0]>=0.95 and scores[i][0]<0.99:    #if reached the finish line
            combined.append( scores[i][0]+ (scores[i][0]/scores[i][1]) ) #sum space+speed
        elif scores[i][0]>=0.99:
            combined.append( scores[i][0]+ (scores[i][0]/scores[i][1])*10 ) #one order of magnitude bonus 
    return combined  # 1-D array of score numbers

Armed with this function, let's find out the best performing cars.

Implement `select_best`, a function taking:
1. a car population, as returned by `initial_population`;
2. scores: the output of `carsim.race` on those cars;
3. fraction: a number between 0 and 1.

`select_best` should rank all the cars according to their score and return only the best ones. The exact number is determined by fraction: if it's .3, the function should return 30% of the original population size, picking only the best individuals.

In [25]:
import collections
def select_best(population, scores, fraction):
    combined = combined_scores(scores)
    d={s:car for s,car in zip(combined,population)}
    
    rank= collections.OrderedDict(sorted(d.items(), reverse=True))
    
    best=list(rank.values())[:int(fraction*len(population))]
    
    return best

### Step 6

Taking inspiration from biology, we have realized some sort of *selection* based on a fitness criterium.

This is only half of our path towards a genetic algorithm. Now we want *evolution*. We want to simulate the mating of two individuals producing an offspring which displays some of the traits of each parent.

Let's reach that goal in small steps. First of all implement the function `mate`. It should take as input two cars and it should generate a third one inheriting part of its genome from car 1 and part from car 2.

**Note**: remember that in *Step 1* you have written two useful functions, `car_genome` and `car_from_genome`.

In [26]:
def mate(car1, car2):
    g1=car_genome(car1)
    g2=car_genome(car2)
    g3=np.zeros(len(g1))
    
    for i in range(len(g1)):  #random inheriting
        r=random.randint(0,1)
        if r==0:
            g3[i]=g1[i]
        elif r==1:
            g3[i]=g2[i]
    
    rcar=random_car()
    rg=car_genome(rcar)
    for i in range( len(g3)//10 ): # 1 mutation every 10 positions
        p=random.randint(0,len(g3)-1)
        g3[p]=rg[p]  #insert mutation from random car at random position
    
    car3=car_from_genome(g3)    
    return car3 

Now make a small improvement to the above implementation: add some random mutation into the mix.

Generate some (small) random values and add those to the genome of the offspring. Which random number should you use? What should the expected range of the random values be?

### Step 7

If you take a close look to the output of the `mate` function updated with mutation, you will notice that it sometimes produces an invalid result. For instance, it may generate a frame length which is longer than 6 meters.

Write a function `adjust_car` which:
1. takes as input a car;
2. if needed, alters its measures to make them compatible with the provided constraints;
3. returns the adjusted car.

For instance, in the above case (frame length > 6 meters) the `adjust_car` should reduce the frame length to 6 meters. All other measures may remain the same, as long as they are themeselves valid.

**Note**: it may be easier to build `adjust_car` from smaller functions. Consider: `adjust_shapes` (which fixes the upper and lower shapes) and `adjust_wheel` (which fixes one wheel at a time).

In [27]:
def adjust_car(car):
    if car[0]<2 or car[0]>6:  #total frame
        car[0]=random.uniform(2,6)
    
    for i in range(0,5):   #upper lower 
        s=car[1][i]+car[2][i]
        if s>=5:
            car[1][i]=4.99-car[2][i]
        d=car[1][i]-car[2][i]
        if d<0.5:
            if car[2][i]>=0.51:
                car[2][i]=car[1][i]-0.5
            else:
                car[1][i]+=0.5
    
    if car[3][1]>2 or car[3][1]<0.1:     #wheel radius
        car[3][1]=random.uniform(2,6)
    if car[4][1]>2 or car[4][1]<0.1:
        car[4][1]=random.uniform(2,6)
    
    if (car[4][0]-car[3][0]<0) or ( (car[4][0] or car[3][0])>car[0] ):  #wheel position
        car[3][0]=random.uniform(0.0,0.6)
        car[4][0]=random.uniform(0.6,1.0)
    
    return car

In [28]:
adjust_car(mate(car1,car2))

[3.242073126292322,
 array([2.49022841, 2.998894  , 2.49022841, 2.49022841, 2.49022841]),
 array([0.43978991, 0.22656109, 0.27409281, 1.54773475, 0.6012544 ]),
 array([0.31396373, 0.45014117]),
 array([0.81904236, 0.92690407])]

### Step 8

Time for the final touches.

Write a `generation` function, to make the car population evolve by a single generation.

It takes an initial population of size P (cars) as a single argument. It should then:
1. make the cars race;
2. collect their scores;
3. select 30% of the best cars;
4. build a new car generation of size P, using:
   1. the 30% best cars selected above, say they are $N$ in number;
   1. $N$ new individuals obtained by picking pairs of best cars at random and making them mate;
   1. $P - 2N$ individuals generated totally at random (using again the function `initial_population`).

In [29]:
def generation(cars):
    scores=carsim.race(cars)
    best=select_best(cars,scores,0.30)  #take the first N best cars (30%)
    rcars=initial_population( len(cars)-2*len(best) ) #takes P-2N random cars
    
    p=list()
    for i in range(len(best)):  #creates N son cars
        s=adjust_car(mate(cars[random.randint(0,len(best)-1)],cars[random.randint(0,len(best)-1)]))
        p.append(s)
    next_gen= p + rcars + best           
    return next_gen

### Step 9

Write `generations`, a functions that runs `generate` a specified number of times, returning the last available car generation.

In [30]:
def generations(cars, gen_num):
    for i in range(gen_num+1):
        cars=generation(cars)
        
    return cars

### Step 10

Everything is finally set.

We make an initial population of 100 cars.

In [31]:
pop1 = initial_population(100)

We select the best individual and simulate its race to get a feeling for how well it behaves.

In [32]:
best = select_best(pop1, carsim.race(pop1), .1)[0]

In [33]:
best

[3.907052913673606,
 array([0.74854664, 0.74854664, 0.74854664, 0.74854664, 0.74854664]),
 array([0.04369492, 0.12476578, 0.19064121, 0.15451226, 0.10366961]),
 array([0.0080988 , 0.96482675]),
 array([0.63756463, 1.42082404])]

In [34]:
carsim.simulate(*best)

We let evolution and selection do their work for 30 generations, and check if they actually introduced an improvement.

In [35]:
popX = generations(pop1, 30)

In [36]:
carsim.race(popX)

array([[9.01523e-02, 6.33333e+00],
       [6.24429e-02, 5.53333e+00],
       [3.64650e-02, 5.83333e+00],
       [5.17562e-02, 5.50000e+00],
       [7.11818e-02, 4.80000e+00],
       [9.34459e-02, 5.80000e+00],
       [9.68216e-03, 4.13333e+00],
       [6.07881e-02, 5.10000e+00],
       [8.13558e-02, 5.70000e+00],
       [6.93949e-02, 5.80000e+00],
       [5.00011e-02, 4.03333e+00],
       [1.56792e-01, 8.20000e+00],
       [1.05288e-01, 5.86667e+00],
       [2.48280e-02, 4.73333e+00],
       [5.16066e-02, 4.73333e+00],
       [5.99430e-02, 5.66667e+00],
       [1.09566e-01, 5.96667e+00],
       [1.41026e-01, 6.40000e+00],
       [3.66070e-02, 5.06667e+00],
       [2.25367e-02, 5.30000e+00],
       [6.83402e-02, 5.36667e+00],
       [3.69688e-02, 4.73333e+00],
       [4.47588e-02, 4.73333e+00],
       [6.17597e-02, 4.03333e+00],
       [4.91744e-02, 5.80000e+00],
       [1.66369e-01, 7.90000e+00],
       [7.06607e-02, 6.20000e+00],
       [8.37068e-02, 5.93333e+00],
       [7.18409e-02,

In [37]:
carsim.simulate(*select_best(popX, carsim.race(popX), .1)[0]);

If at this point we haven't yet reached a satisfactory design, we let the system evolve for more generations.

In [38]:
popX = generations(popX, 30)

Notice like in the above code we evolved population X which was already the result of an evolution. If `popX` was originally 30 generation old, now it got 60 generations old.

### Step 11

When you get a working car, try changing the track profile.

In [39]:
carsim.new_track()

In [42]:
carsim.simulate(*select_best(popX, carsim.race(popX), .1)[0]);

In [41]:
b = select_best(popX, carsim.race(popX), .2)[:10]
carsim.race(b)

array([[ 1.     ,  9.46667],
       [ 1.     ,  9.83333],
       [ 1.     , 10.0333 ],
       [ 1.     , 10.0667 ],
       [ 1.     , 10.1    ],
       [ 1.     , 10.3333 ],
       [ 1.     , 10.4667 ],
       [ 1.     , 10.5333 ],
       [ 1.     , 10.5667 ],
       [ 1.     , 10.6333 ]])

Is you car still fit for the new track? What does it mean if the car was good previously, but now that the track has changed is no longer working?

The car still fits well on the new track, as well as the other 10 best cars. I would nonetheless expect an eventual reduction in fitnes since we made the cars evolve and be selected for a certain environment or track. 
Changing it thus, there is no guarantee that the features selected for the previous environment will be advantageous also for the new one.